In [2]:
!pip install chromadb sentence-transformers pypdf transformers accelerate torch -q

In [3]:
import chromadb
print("ChromaDB Loaded Successfully")

ChromaDB Loaded Successfully


In [4]:
!pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk opentelemetry-proto

!pip install \
chromadb==0.5.5 \
sentence-transformers==2.7.0 \
pypdf==4.2.0 \
transformers==4.41.2 \
accelerate==0.31.0 -q

Found existing installation: chromadb 0.5.5
Uninstalling chromadb-0.5.5:
  Successfully uninstalled chromadb-0.5.5
Found existing installation: opentelemetry-api 1.42.1
Uninstalling opentelemetry-api-1.42.1:
  Successfully uninstalled opentelemetry-api-1.42.1
Found existing installation: opentelemetry-sdk 1.42.1
Uninstalling opentelemetry-sdk-1.42.1:
  Successfully uninstalled opentelemetry-sdk-1.42.1
Found existing installation: opentelemetry-proto 1.42.1
Uninstalling opentelemetry-proto-1.42.1:
  Successfully uninstalled opentelemetry-proto-1.42.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38

In [5]:
!pip uninstall -y numpy scipy scikit-learn sentence-transformers
!pip install numpy==1.26.4 scipy==1.13.1 scikit-learn==1.5.0 sentence-transformers==2.7.0 -q

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.13.1
Uninstalling scipy-1.13.1:
  Successfully uninstalled scipy-1.13.1
Found existing installation: scikit-learn 1.5.0
Uninstalling scikit-learn-1.5.0:
  Successfully uninstalled scikit-learn-1.5.0
Found existing installation: sentence-transformers 2.7.0
Uninstalling sentence-transformers-2.7.0:
  Successfully uninstalled sentence-transformers-2.7.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 whic

In [6]:
from google.colab import files

uploaded = files.upload()

Saving resume_adhidhya.pdf to resume_adhidhya (2).pdf
Saving hemantth_resume.pdf to hemantth_resume (2).pdf
Saving RESUME.pdf to RESUME (2).pdf


In [7]:
from pypdf import PdfReader

documents = []

pdf_files = [
    "resume_adhidhya.pdf",
    "hemantth_resume.pdf",
    "RESUME.pdf"
]

for file in pdf_files:
    reader = PdfReader(file)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    documents.append({
        "id": file,
        "text": text
    })

print("Loaded:", len(documents), "resumes")

Loaded: 3 resumes


/usr/local/lib/python3.12/dist-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [doc["text"] for doc in documents]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="resume_rag"
)

collection.add(
    ids=[doc["id"] for doc in documents],
    documents=texts,
    embeddings=embeddings.tolist()
)

print("Stored in ChromaDB")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Stored in ChromaDB


In [10]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLM Loaded")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

LLM Loaded


In [11]:
def retrieve_resume(query):

    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True
    )

    result = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=2
    )

    return result["documents"][0]

In [12]:
def ask_resume_bot(question):

    retrieved_docs = retrieve_resume(question)

    context = "\n\n".join(retrieved_docs)

    prompt = f"""
You are an HR assistant.

Use the resume information below.

Resume Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [13]:
question = "Which candidate has Python and Machine Learning skills?"

answer = ask_resume_bot(question)

print(answer)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(



You are an HR assistant.

Use the resume information below.

Resume Context:
PARIDHI RAMESH KUMAR
AI  ENGINEER
Aspiring AI engineer with a passion for Python programming and development models. Keen to learn and apply
machine learning techniques, build intelligent systems, and contribute to innovative AI-driven solutions. Eager to work
in dynamic environments that encourage innovation, collaboration, and professional growth.
CAREER OBJECTIVE 
EDUCATION
 Karpagam College of Engineering,Coimbatore
Sri Chaitanya Techno School,Erode 
B.TECH AI & DS
Higher secondary 2024 - 2028
2022 - 20248.15 %  (2025)
77.6  %
SKILLS
Leadership
ACHIEVEMENTS TeamworkCommunication
Problem SolvingInnovation 
Collaboration+91 638 158 4434 paridhi2810@gmail.com Erode, 638011
Velalar Vidhyalayaa Senior Secondary School,Erode 
81.6 %2021 - 2022
Senior secondary 
TECHNICAL SKILLS Python , Java ,  HTML , CSS , Javascript
PROJECTS 
virtual web app that lets users explore, share announcements, events, and messages i